# Results — dissertation figures and tables

Regenerates every Chapter 5 figure/table from `results/runs/*` and
`results/latency.csv`. Run after the 12-run grid has been trained and each
checkpoint evaluated with `python -m src.evaluate ... --split val`
(README workflow steps 6-9).

Run groups: **A** transformer+aug, **B** transformer no-aug, **C** LSTM+aug,
**D** LSTM no-aug; three seeds (42/43/44) each.

In [ ]:
from pathlib import Path
import sys
import subprocess

REPO = Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

RUNS = REPO / 'results' / 'runs'
if not RUNS.exists():
    raise FileNotFoundError(
        f'{RUNS} not found - train the 12-run grid first (README workflow step 6).')

run_dirs = sorted(d for d in RUNS.iterdir() if (d / 'metrics.csv').exists())
print(f'{len(run_dirs)} runs found: {[d.name for d in run_dirs]}')


def parse_run(name: str):
    arch, aug, seed = name.rsplit('_', 2)
    return arch, aug == 'aug', int(seed.lstrip('s'))

## Learning curves (Figure for §5.1 Training behaviour)

Validation accuracy per epoch for all runs; solid = augmented, dashed = no
augmentation. Destination: Chapter 5, §5.1.

In [ ]:
records = []
curves = {}
for d in run_dirs:
    m = pd.read_csv(d / 'metrics.csv')
    arch, aug, seed = parse_run(d.name)
    curves[d.name] = m
    records.append({'run_id': d.name, 'arch': arch, 'augment': aug, 'seed': seed,
                    'best_val_acc': m['val_acc'].max(),
                    'best_epoch': int(m.loc[m['val_acc'].idxmax(), 'epoch'])})
runs = pd.DataFrame(records)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for name, m in curves.items():
    arch, aug, seed = parse_run(name)
    ax = axes[0] if arch == 'transformer' else axes[1]
    ax.plot(m['epoch'], m['val_acc'], alpha=0.8,
            ls='-' if aug else '--', label=name)
for ax, title in zip(axes, ['transformer', 'lstm']):
    ax.set_title(title)
    ax.set_xlabel('epoch')
    ax.legend(fontsize=7)
axes[0].set_ylabel('val accuracy')
fig.suptitle('Validation accuracy over 80 epochs')
plt.tight_layout()
plt.show()
runs.sort_values(['arch', 'augment', 'seed'])

## Mean ± std over seeds per run group (Table for §5.1)

Best validation accuracy aggregated over seeds 42/43/44 for groups A-D.
Destination: Chapter 5, §5.1 (headline results table).

In [ ]:
GROUPS = {'A': ('transformer', True), 'B': ('transformer', False),
          'C': ('lstm', True), 'D': ('lstm', False)}
rows = []
for g, (arch, aug) in GROUPS.items():
    sel = runs[(runs['arch'] == arch) & (runs['augment'] == aug)]['best_val_acc']
    rows.append({'group': g, 'arch': arch, 'augment': aug, 'n_seeds': len(sel),
                 'val_acc': f'{sel.mean():.3f} +/- {sel.std(ddof=1):.3f}'
                            if len(sel) > 1 else f'{sel.mean():.3f}'})
group_table = pd.DataFrame(rows).set_index('group')
group_table

## RQ1 — Transformer vs LSTM (Table for §5.2)

Top-1 / top-5 / macro-F1 recomputed from each run's `eval_val/predictions.csv`,
aggregated over seeds for the augmented condition. Destination: Chapter 5, §5.2
(architecture comparison, RQ1).

In [ ]:
def eval_metrics(run_dir: Path, split: str = 'val'):
    p = run_dir / f'eval_{split}' / 'predictions.csv'
    if not p.exists():
        return None
    df = pd.read_csv(p)
    top5_cols = [c for c in df.columns if c.startswith('top5')]
    top5 = (df.apply(lambda r: r['true'] in [r[c] for c in top5_cols], axis=1).mean()
            if top5_cols else np.nan)
    return {'top1': df['correct'].mean(), 'top5': top5,
            'macro_f1': f1_score(df['true'], df['pred'], average='macro'),
            'predictions': p}


rows = []
for _, r in runs.iterrows():
    m = eval_metrics(RUNS / r['run_id'])
    if m is not None:
        rows.append({'run_id': r['run_id'], 'arch': r['arch'],
                     'augment': r['augment'], 'seed': r['seed'],
                     'top1': m['top1'], 'top5': m['top5'],
                     'macro_f1': m['macro_f1']})
ev = pd.DataFrame(rows)
if ev.empty:
    print('No eval_val outputs found - run `python -m src.evaluate '
          '--checkpoint results/runs/<run>/best.pt --split val` per run first.')
else:
    rq1 = (ev[ev['augment']]
           .groupby('arch')[['top1', 'top5', 'macro_f1']]
           .agg(['mean', 'std']).round(3))
    display(rq1)

## RQ2 — Augmentation ablation deltas (Table for §5.3)

Mean top-1 with vs without augmentation per architecture, and the delta.
Destination: Chapter 5, §5.3 (augmentation ablation, RQ2).

In [ ]:
if not ev.empty:
    pivot = ev.groupby(['arch', 'augment'])['top1'].mean().unstack()
    pivot.columns = ['noaug', 'aug']
    pivot['delta_aug'] = pivot['aug'] - pivot['noaug']
    display(pivot.round(3))

## Confusion matrices — best transformer and best LSTM (Figures for §5.2)

Loaded from each best run's `eval_val/confusion_matrix.csv`. Destination:
Chapter 5, §5.2 (error analysis).

In [ ]:
def best_run(arch: str):
    if ev.empty:
        return None
    sel = ev[(ev['arch'] == arch) & (ev['augment'])]
    if sel.empty:
        sel = ev[ev['arch'] == arch]
    return None if sel.empty else sel.loc[sel['top1'].idxmax(), 'run_id']


for arch in ['transformer', 'lstm']:
    rid = best_run(arch)
    if rid is None:
        print(f'no evaluated {arch} runs')
        continue
    cm_path = RUNS / rid / 'eval_val' / 'confusion_matrix.csv'
    if not cm_path.exists():
        print(f'{cm_path} missing')
        continue
    cm = pd.read_csv(cm_path, index_col=0)
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cm.values, cmap='Blues')
    ax.set_title(f'Confusion matrix - {rid} (val)')
    ax.set_xlabel('predicted')
    ax.set_ylabel('true')
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()

## Latency (Table for §5.4)

Per-stage median/p95 and end-to-end fps from `results/latency.csv`
(written by `python -m src.benchmark`). Destination: Chapter 5, §5.4
(real-time criterion: >= 15 fps, < 100 ms per window).

In [ ]:
lat_path = REPO / 'results' / 'latency.csv'
if lat_path.exists():
    display(pd.read_csv(lat_path))
else:
    print('results/latency.csv not found - run `python -m src.benchmark` first '
          '(README workflow step 9).')

## McNemar's test — best transformer vs best LSTM (§5.2)

Exact binomial McNemar on paired per-clip correctness (alpha = 0.05), via the
project CLI so the dissertation quotes exactly what the tooling reports.
Destination: Chapter 5, §5.2 (significance of the architecture difference).

In [ ]:
best_t, best_l = best_run('transformer'), best_run('lstm')
if best_t and best_l:
    preds_a = RUNS / best_t / 'eval_val' / 'predictions.csv'
    preds_b = RUNS / best_l / 'eval_val' / 'predictions.csv'
    result = subprocess.run(
        [sys.executable, '-m', 'src.evaluate', '--mcnemar',
         str(preds_a), str(preds_b)],
        cwd=str(REPO), capture_output=True, text=True)
    print(f'A = {best_t}\nB = {best_l}\n')
    print(result.stdout or result.stderr)
else:
    print('need at least one evaluated transformer and one evaluated lstm run')